In [1]:
import pandas as pd
from transformers import T5Tokenizer
from transformers import Trainer,TrainingArguments,T5ForConditionalGeneration


In [2]:
train_data = pd.read_csv("samsum-train.csv")

In [3]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [6]:
train_data.head(3)

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...


In [7]:
train_data.shape

(14732, 3)

In [8]:
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)

In [9]:
val_data = val_data.sample(n=500,random_state=42).reset_index(drop=True)


# Data Pre-processing

In [10]:
import re
def clean_data(text):
    text = re.sub(r"\r\n"," ",text)# lines
    text = re.sub(r"\s+"," ",text)# space
    text = re.sub(r"<,*?>"," ",text)# html tags <p> <h1>
    text = text.strip().lower()
    return text

In [11]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


# tokenize


In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


In [13]:
#raw data => tokenized inputs for fine tuninig

def tokenize(data):
   inputs= tokenizer(data["dialogue"],padding="max_length",max_length = 512,truncation=True)
   targets= tokenizer(data["summary"],padding="max_length",max_length = 150,truncation=True)
   inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
   return inputs

In [14]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize,axis = 1).tolist()


In [15]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 2, 11966, 834, 9269, 3155, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

input_ids
attention mask
labels - target => summary token

In [16]:
# Fine tuning - Conditional Generation
model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [17]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device=torch.device("cpu")

print("device:",device)
model.to(device)

device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

# Training Argumnets

In [18]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay=0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

   eval_strategy="epoch",
   save_strategy = "epoch",

    warmup_steps = 500
    # 0 => lr default
)


In [19]:
trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.636266,0.382097
2,0.395675,0.359097
3,0.372619,0.353827
4,0.361559,0.349980
5,0.355064,0.349178
6,0.350450,0.348686


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9119386189778645, metrics={'train_runtime': 2060.0921, 'train_samples_per_second': 11.65, 'train_steps_per_second': 1.456, 'total_flos': 3248203235328000.0, 'train_loss': 0.9119386189778645, 'epoch': 6.0})

In [21]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [22]:
model.from_pretrained("./saved_summary_model")
tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5Tokenizer(name_or_path='./saved_summary_model', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False

 Test the core logic for summerizationm

In [23]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) #clean
    #tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length=1000,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    # generate the summary => token ids
    model.to(device,)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length=500,
        num_beams =8,
        early_stopping =True
    )
    #token ids convert to summary =>decoding
    summary = tokenizer.decode(targets[0],skip_special_tokens=True) #EOS, SEP
    return summary
    

In [24]:
test_dialogue=""""""
summary = summarize_dialogue(test_dialogue)
print("Summary:",summary)

Summary: 
